# Interactive model query playground

Ask `models/qwen3-4b-qlora` (Qwen3-4B + QLoRA adapter, Stage 2 of the 4B pivot) questions directly instead of reading aggregate metrics -- useful for building intuition about *how* it's wrong, not just how often.

Five convenience functions, one per task the model was trained on:
- `ask_name2id(game_name)` -- name -> semantic ID
- `ask_id2name(sid_string)` -- semantic ID -> name
- `ask_sequential(history_sids)` -- play history -> next item's semantic ID
- `ask_asy(history_sids)` -- play history -> next item's name
- `ask_similar(sid_string)` -- one item -> a similar item's semantic ID

Each takes `constrained=True` to force the output through the trie (guaranteed a real catalog item -- see `src/constrained_decoding.py`) instead of raw greedy decoding.

Catalog lookup helpers (`find_items`, `sid_for`, `name_for`) let you get ground truth to compare against, and `round_trip(game_name)` chains name->ID->name to see whether the two directions agree with each other even when neither matches the true value.

In [ ]:
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

PROJECT_ROOT = Path('..')
sys.path.insert(0, str((PROJECT_ROOT / 'src').resolve()))

from constrained_decoding import build_name_trie, build_sid_trie, constrained_generate, load_catalog

BASE_MODEL_NAME = 'Qwen/Qwen3-4B'
ADAPTER_PATH = (PROJECT_ROOT / 'models' / 'qwen3-4b-qlora').resolve().as_posix()

## Load the model, tokenizer, and catalog

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, llm_int8_skip_modules=['lm_head'],
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, dtype=torch.bfloat16, quantization_config=quantization_config,
)
base_model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

catalog = load_catalog(PROJECT_ROOT)  # restricted to the 8,563 items the model was trained on
sid_trie = build_sid_trie(tokenizer, catalog)
name_trie = build_name_trie(tokenizer, catalog)
print(f'Model loaded. Catalog: {len(catalog)} items.')

## Catalog lookup helpers (ground truth, not the model)

In [4]:
from constrained_decoding import item_description, semantic_id_to_tokens

_by_name = {row['Name'].lower(): row for row in catalog.iter_rows(named=True)}

def find_items(substring, limit=15):
    """Search the catalog by (case-insensitive) name substring."""
    substring = substring.lower()
    matches = [row for row in catalog.iter_rows(named=True) if substring in row['Name'].lower()]
    for row in matches[:limit]:
        print(f"{row['Name']!r:60} {semantic_id_to_tokens(row['semantic_ids'])}")
    if len(matches) > limit:
        print(f'... and {len(matches) - limit} more')
    return matches

def sid_for(game_name):
    """The REAL semantic ID for an exact catalog title (ground truth)."""
    row = _by_name.get(game_name.lower())
    return semantic_id_to_tokens(row['semantic_ids']) if row else None

def name_for(sid_string):
    """The REAL name+genres for a semantic ID (ground truth)."""
    target = sid_string.strip()
    for row in catalog.iter_rows(named=True):
        if semantic_id_to_tokens(row['semantic_ids']) == target:
            return item_description(row['Name'], row['Genres'])
    return None

## Example: searching the catalog

The model only knows the ~8,563 items here -- if a game you try isn't in this list, that's not a grounding failure, it's an out-of-scope question.

In [126]:
find_items('Sonic')

'Sonic & SEGA All-Stars Racing'                              <|sid_start|><|sid_L0_54|><|sid_L1_180|><|sid_L2_20|><|sid_L3_0|><|sid_end|>
'Sonic & All-Stars Racing Transformed Collection'            <|sid_start|><|sid_L0_54|><|sid_L1_230|><|sid_L2_220|><|sid_L3_0|><|sid_end|>
'Sonic the Hedgehog 4 - Episode II'                          <|sid_start|><|sid_L0_246|><|sid_L1_152|><|sid_L2_130|><|sid_L3_0|><|sid_end|>
'Sonic Adventure 2'                                          <|sid_start|><|sid_L0_246|><|sid_L1_230|><|sid_L2_174|><|sid_L3_0|><|sid_end|>
'Sonic Generations Collection'                               <|sid_start|><|sid_L0_151|><|sid_L1_110|><|sid_L2_178|><|sid_L3_0|><|sid_end|>
'Sonic Adventure DX'                                         <|sid_start|><|sid_L0_239|><|sid_L1_152|><|sid_L2_127|><|sid_L3_0|><|sid_end|>
'Sonicomi'                                                   <|sid_start|><|sid_L0_118|><|sid_L1_214|><|sid_L2_18|><|sid_L3_0|><|sid_end|>
'Sonic Lost World'      

[{'id': '34190',
  'semantic_ids': [54, 180, 20, 0],
  'semantic_id_0': 54,
  'semantic_id_1': 180,
  'semantic_id_2': 20,
  'semantic_id_3': 0,
  'Name': 'Sonic & SEGA All-Stars Racing',
  'Genres': 'Racing'},
 {'id': '212480',
  'semantic_ids': [54, 230, 220, 0],
  'semantic_id_0': 54,
  'semantic_id_1': 230,
  'semantic_id_2': 220,
  'semantic_id_3': 0,
  'Name': 'Sonic & All-Stars Racing Transformed Collection',
  'Genres': 'Racing,Sports'},
 {'id': '203650',
  'semantic_ids': [246, 152, 130, 0],
  'semantic_id_0': 246,
  'semantic_id_1': 152,
  'semantic_id_2': 130,
  'semantic_id_3': 0,
  'Name': 'Sonic the Hedgehog 4 - Episode II',
  'Genres': 'Adventure'},
 {'id': '213610',
  'semantic_ids': [246, 230, 174, 0],
  'semantic_id_0': 246,
  'semantic_id_1': 230,
  'semantic_id_2': 174,
  'semantic_id_3': 0,
  'Name': 'Sonic Adventure 2',
  'Genres': 'Action'},
 {'id': '71340',
  'semantic_ids': [151, 110, 178, 0],
  'semantic_id_0': 151,
  'semantic_id_1': 110,
  'semantic_id_2': 1

## Ask the model

In [6]:
@torch.no_grad()
def _generate(instruction, input_text, trie=None, max_new_tokens=32, temperature=None):
    messages = [{'role': 'user', 'content': f'{instruction}\n{input_text}'}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if trie is not None:
        return constrained_generate(model, tokenizer, prompt, trie, max_new_tokens=max_new_tokens, temperature=temperature)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    sampling_kwargs = {'do_sample': False} if temperature is None else {'do_sample': True, 'temperature': temperature}
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        **sampling_kwargs,
    )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False).replace(tokenizer.eos_token, '').strip()

def ask_name2id(game_name, constrained=False, temperature=None):
    return _generate('What is the semantic ID for this game?', game_name, sid_trie if constrained else None, temperature=temperature)

def ask_id2name(sid_string, constrained=False, temperature=None):
    return _generate('What game does this semantic ID represent?', sid_string, name_trie if constrained else None, temperature=temperature)

def ask_sequential(history_sids, constrained=False, temperature=None):
    """history_sids: list of semantic-ID strings, most-played first."""
    return _generate(
        "Given a user's game history, ordered from most to least played, predict the semantic ID of the next game they are likely to enjoy.",
        ' '.join(history_sids), sid_trie if constrained else None, temperature=temperature,
    )

def ask_asy(history_sids, constrained=False, temperature=None):
    """Same as ask_sequential, but asks for the next game's NAME instead of its ID."""
    return _generate(
        "Given a user's game history, ordered from most to least played, predict the name of the next game they are likely to enjoy.",
        ' '.join(history_sids), name_trie if constrained else None, temperature=temperature,
    )

def ask_similar(sid_string, constrained=False, temperature=None):
    return _generate('A player enjoyed this game. Suggest another game they would likely also enjoy.', sid_string, sid_trie if constrained else None, temperature=temperature)

## Round-trip check: name -> ID -> name

Feeds the model's own name->ID answer back into id->name. Shows whether the two directions agree with each other, independent of whether either matches the true catalog entry.

In [7]:
def round_trip(game_name, constrained=False, temperature=None):
    true_sid = sid_for(game_name)
    predicted_sid = ask_name2id(game_name, constrained=constrained, temperature=temperature)
    predicted_name_back = ask_id2name(predicted_sid, constrained=constrained, temperature=temperature)
    print(f'input name:          {game_name!r}')
    print(f'true sid:            {true_sid!r}')
    print(f'predicted sid:       {predicted_sid!r}')
    print(f'  -> fed back in:    {predicted_name_back!r}')
    print(f'sid exact match:     {predicted_sid == true_sid}')

## Try it

In [37]:
round_trip('Half-Life 2', temperature=1.0, constrained=True)

input name:          'Half-Life 2'
true sid:            '<|sid_start|><|sid_L0_245|><|sid_L1_139|><|sid_L2_254|><|sid_L3_0|><|sid_end|>'
predicted sid:       '<|sid_start|><|sid_L0_245|><|sid_L1_181|><|sid_L2_104|><|sid_L3_0|><|sid_end|>'
  -> fed back in:    'Hard Reset Extended Edition — Action, Indie'
sid exact match:     False


In [9]:
print('raw:        ', ask_name2id('Portal 2'))
print('constrained:', ask_name2id('Portal 2', constrained=True))
print('true sid:   ', sid_for('Portal 2'))

raw:         <|sid_start|><|sid_L0_246|><|sid_L1_82|><|sid_L2_104|><|sid_L3_0|><|sid_end|>
constrained: <|sid_start|><|sid_L0_246|><|sid_L1_82|><|sid_L2_104|><|sid_L3_0|><|sid_end|>
true sid:    <|sid_start|><|sid_L0_246|><|sid_L1_82|><|sid_L2_104|><|sid_L3_0|><|sid_end|>


## Temperature

All `ask_*`/`round_trip` functions default to greedy (deterministic) decoding. Pass `temperature=` to sample instead -- e.g. `0.7` for some variety, higher for more, lower (close to 0) to approach greedy. Only takes effect when set; omit it for the default greedy behavior used everywhere else in this notebook.

In [10]:
# Same question, sampled 3 times at temperature=0.8 -- greedy would give the same answer every time.
for _ in range(3):
    print(ask_name2id('Portal 2', temperature=0.8))

<|sid_start|><|sid_L0_246|><|sid_L1_82|><|sid_L2_104|><|sid_L3_0|><|sid_end|>
<|sid_start|><|sid_L0_230|><|sid_L1_244|><|sid_L2_249|><|sid_L3_0|><|sid_end|>
<|sid_start|><|sid_L0_246|><|sid_L1_82|><|sid_L2_86|><|sid_L3_0|><|sid_end|>


In [11]:
# Sequential: pick a few real items and see what it predicts next.
history = [row['semantic_ids'] for row in catalog.iter_rows(named=True)][:3]
from constrained_decoding import semantic_id_to_tokens as _sid_to_tok
history_sids = [_sid_to_tok(h) for h in history]
print('history:', history_sids)
print('next (raw sid):  ', ask_sequential(history_sids))
print('next (raw name): ', ask_asy(history_sids))

history: ['<|sid_start|><|sid_L0_208|><|sid_L1_172|><|sid_L2_94|><|sid_L3_0|><|sid_end|>', '<|sid_start|><|sid_L0_198|><|sid_L1_32|><|sid_L2_161|><|sid_L3_0|><|sid_end|>', '<|sid_start|><|sid_L0_161|><|sid_L1_223|><|sid_L2_211|><|sid_L3_0|><|sid_end|>']
next (raw sid):   <|sid_start|><|sid_L0_198|><|sid_L1_32|><|sid_L2_111|><|sid_L3_0|><|sid_end|>
next (raw name):  The Witcher 2: Assassins of Kings Enhanced Edition — RPG


## Your turn

Add cells below and call `ask_name2id`, `ask_id2name`, `ask_sequential`, `ask_asy`, `ask_similar`, `round_trip`, `find_items`, `sid_for`, `name_for` with whatever you want to try.

In [25]:
round_trip('Counter-Strike: Condition Zero', constrained=True)

input name:          'Counter-Strike: Condition Zero'
true sid:            '<|sid_start|><|sid_L0_115|><|sid_L1_251|><|sid_L2_98|><|sid_L3_1|><|sid_end|>'
predicted sid:       '<|sid_start|><|sid_L0_115|><|sid_L1_251|><|sid_L2_98|><|sid_L3_0|><|sid_end|>'
  -> fed back in:    'Counter-Strike: Condition Zero — Action'
sid exact match:     False


In [61]:
round_trip('Street Fighter® IV')

input name:          'Street Fighter® IV'
true sid:            '<|sid_start|><|sid_L0_23|><|sid_L1_62|><|sid_L2_105|><|sid_L3_0|><|sid_end|>'
predicted sid:       '<|sid_start|><|sid_L0_23|><|sid_L1_118|><|sid_L2_221|><|sid_L3_0|><|sid_end|>'
  -> fed back in:    'Street Fighter X Tekken — Action'
sid exact match:     False


In [82]:
ask_similar('<|sid_start|><|sid_L0_115|><|sid_L1_139|><|sid_L2_83|><|sid_L3_0|><|sid_end|>', temperature=1.0, constrained=True)

'<|sid_start|><|sid_L0_115|><|sid_L1_76|><|sid_L2_148|><|sid_L3_0|><|sid_end|>'

In [84]:
ask_asy(['<|sid_start|><|sid_L0_23|><|sid_L1_110|><|sid_L2_170|><|sid_L3_0|><|sid_end|>', '<|sid_start|><|sid_L0_115|><|sid_L1_139|><|sid_L2_83|><|sid_L3_0|><|sid_end|>'], temperature=1.0, constrained=True)

'Tom Clancy’s Splinter Cell Blacklist — Action, Adventure'

In [139]:
_generate('You are an expert in game recommendations. Make sure your recommendations are relevant and helpful.', 
                 'Recommend games similar to Mortal Kombat 1', sid_trie, temperature=0.8)

'<|sid_start|><|sid_L0_23|><|sid_L1_107|><|sid_L2_73|><|sid_L3_0|><|sid_end|>'